In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 706, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 706 (delta 42), reused 24 (delta 13), pack-reused 641 (from 2)
Receiving objects: 100% (706/706), 595.19 KiB | 6.54 MiB/s, done.
Resolving deltas: 100% (475/475), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [4]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2983, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27880, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38127, 38)
Loaded: raw_syp_simas_sales_bills.csv -> (13021, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154087, 41)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50269, 49)
Loaded: raw_hq_icmas_products.csv -> (114973, 94)
Loaded: raw_hq_sidet_sales_lines.csv -> (733591, 38)
Loaded: raw_hq_simas_sales_bills.csv -> (276119, 49)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_armas_receivable.csv -> (2228, 20)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13875, 32)


In [6]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/statement"

In [7]:
!pip install xlrd

In [8]:
import os
import pandas as pd

data_statement = {}

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    ext = os.path.splitext(file)[1].lower()

    try:
        if ext == ".csv":
            # text file
            last_err = None
            for enc in ["utf-8-sig", "cp874", "cp1252", "latin1"]:
                try:
                    df = pd.read_csv(path, skiprows=10, encoding=enc, low_memory=False)
                    source = f"csv-{enc}"
                    break
                except Exception as e:
                    last_err = e
            else:
                raise last_err

        elif ext == ".xlsx":
            # modern Excel
            df = pd.read_excel(path, header=10, engine="openpyxl")
            source = "xlsx-openpyxl"

        elif ext == ".xls":
            # old Excel binary format
            df = pd.read_excel(path, header=10, engine="xlrd")
            source = "xls-xlrd"

        else:
            continue

        for col in ["BCODE", "ITEMNO", "BILLNO"]:
            if col in df.columns:
                df[col] = df[col].astype("string")

        key = os.path.splitext(file)[0]
        data_statement[key] = df
        print(f"Loaded: {file} -> {df.shape} ({source})")

    except Exception as e:
        print(f"Failed: {file} -> {type(e).__name__}: {e}")

Loaded: raw_statement_6184.xls -> (31, 9) (xls-xlrd)
Loaded: raw_statement_3557.csv -> (138, 13) (csv-utf-8-sig)
Loaded: raw_statement_7236.csv -> (92, 13) (csv-utf-8-sig)
Loaded: raw_statement_1139.xls -> (42, 9) (xls-xlrd)
Loaded: raw_statement_0393.csv -> (67, 13) (csv-utf-8-sig)


In [87]:
import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)


<module 'src.kcw.utils' from '/content/kcw-analytics/src/kcw/utils.py'>

In [88]:
df_3557 = data_statement["raw_statement_3557"].copy()

In [89]:
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()
df_pvmas = data["raw_hq_pvmas_notes_vouchers.csv"].copy()

In [90]:
import pandas as pd

def filter_last_1year(df, date_col="BILLDATE"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    cutoff = pd.Timestamp.today() - pd.DateOffset(years=1)

    return df[df[date_col] >= cutoff]

df_pimas = filter_last_1year(df_pimas)
df_pvmas = filter_last_1year(df_pvmas, date_col="VOUCDATE")

In [91]:
import pandas as pd

def match_billno_candidates_by_amount(
    df_source: pd.DataFrame,
    df_statement: pd.DataFrame,
    *,
    source_name: str,
    source_amount_col: str = "AMOUNT",
    statement_amount_col: str = "Amount",
    source_billno_col: str = "BILLNO",
    source_acctname_col: str | None = "ACCTNAME",
    source_date_col: str | None = None,
    tolerance: float = 1.0,
    max_matches: int = 5,
    out_col: str | None = None,
    sort_by_amount_diff: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    For each statement row, find up to max_matches source rows within +/- tolerance
    on amount, then output all matches into one text cell.

    Example cell output:
        BILLNO1 : ACCTNAME1
        BILLNO2 : ACCTNAME2
        BILLNO3 : ACCTNAME3
    """

    source = df_source.copy()
    stmt = df_statement.copy() if copy else df_statement

    source[source_amount_col] = pd.to_numeric(
        source[source_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    stmt[statement_amount_col] = pd.to_numeric(
        stmt[statement_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    if source_date_col is not None and source_date_col in source.columns:
        source[source_date_col] = pd.to_datetime(source[source_date_col], errors="coerce")

    stmt["_row_id"] = range(len(stmt))

    source_valid = source.dropna(subset=[source_amount_col]).copy()
    stmt_valid = stmt.dropna(subset=[statement_amount_col]).copy()

    if out_col is None:
        out_col = f"{source_name}.MATCH_CANDIDATES"

    def build_candidate_text(stmt_amount):
        candidates = source_valid[
            source_valid[source_amount_col].sub(stmt_amount).abs() <= tolerance
        ].copy()

        if candidates.empty:
            return pd.NA

        candidates["_amount_diff"] = (candidates[source_amount_col] - stmt_amount).abs()

        sort_cols = []
        ascending = []

        # latest date first
        if source_date_col is not None and source_date_col in candidates.columns:
            sort_cols.append(source_date_col)
            ascending.append(False)

        # then smallest amount diff
        if sort_by_amount_diff:
            sort_cols.append("_amount_diff")
            ascending.append(True)

        # then stable tie-break by billno
        if source_billno_col in candidates.columns:
            sort_cols.append(source_billno_col)
            ascending.append(True)

        if sort_cols:
            candidates = candidates.sort_values(sort_cols, ascending=ascending)

        candidates = candidates.head(max_matches)

        lines = []
        for _, row in candidates.iterrows():
            billno = row.get(source_billno_col, "")
            acctname = row.get(source_acctname_col, "") if source_acctname_col else ""

            billno = "" if pd.isna(billno) else str(billno)
            acctname = "" if pd.isna(acctname) else str(acctname)

            if source_acctname_col:
                lines.append(f"{billno} : {acctname}")
            else:
                lines.append(billno)

        return "\n".join(lines) if lines else pd.NA

    matched = stmt_valid[["_row_id", statement_amount_col]].copy()
    matched[out_col] = matched[statement_amount_col].apply(build_candidate_text)

    out = stmt.merge(
        matched[["_row_id", out_col]],
        on="_row_id",
        how="left",
    )

    out = out.sort_values("_row_id").drop(columns=["_row_id"])

    if verbose:
        print(
            f"[match_billno_candidates_by_amount] "
            f"matched_rows={out[out_col].notna().sum():,}/{len(out):,}"
        )

    return out

import pandas as pd

def match_multiple_sources_candidates_by_amount(
    df_statement: pd.DataFrame,
    source_configs: list[dict],
    *,
    statement_amount_col: str = "Amount",
    tolerance: float = 1.0,
    max_matches: int = 5,
) -> pd.DataFrame:
    """
    Apply candidate-list amount matching repeatedly using a list of source configs.

    Each config can look like:
    {
        "df_source": df_pvmas,
        "source_name": "vouchers",
        "source_amount_col": "BILLAMT",
        "source_billno_col": "VOUCNO",
        "source_acctname_col": "ACCTNAME",
        "statement_amount_col": "ถอนเงิน",     # optional
        "tolerance": 1.0,                      # optional
        "max_matches": 5,                      # optional
        "out_col": "voucher_candidates",       # optional
    }
    """
    out = df_statement.copy()

    for cfg in source_configs:
        out = match_billno_candidates_by_amount(
            df_source=cfg["df_source"],
            df_statement=out,
            source_name=cfg["source_name"],
            source_amount_col=cfg.get("source_amount_col", "AMOUNT"),
            statement_amount_col=cfg.get("statement_amount_col", statement_amount_col),
            source_billno_col=cfg.get("source_billno_col", "BILLNO"),
            source_acctname_col=cfg.get("source_acctname_col", "ACCTNAME"),
            source_date_col=cfg.get("source_date_col"),
            tolerance=cfg.get("tolerance", tolerance),
            max_matches=cfg.get("max_matches", max_matches),
            out_col=cfg.get("out_col"),
            sort_by_amount_diff=cfg.get("sort_by_amount_diff", True),
            copy=False,
            verbose=cfg.get("verbose", True),
        )

    return out

In [92]:
import pandas as pd

def build_statement_datetime(
    df: pd.DataFrame,
    *,
    date_col: str = "วันที่",
    time_col: str = "เวลา/ วันที่มีผล",
    out_col: str = "date",
    drop_original: bool = False,
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Merge Thai bank statement date + time columns into a single datetime column.

    Example input:
        วันที่           เวลา/ วันที่มีผล
        02-02-26        08:32
        02-02-26        NaN

    Output:
        date
        2026-02-02 08:32
        2026-02-02 00:00
    """

    df = df.copy() if copy else df

    # clean column names
    df.columns = df.columns.str.strip()

    # remove Excel unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    if date_col not in df.columns:
        raise ValueError(f"{date_col} not found in dataframe")

    if time_col not in df.columns:
        raise ValueError(f"{time_col} not found in dataframe")

    df[out_col] = pd.to_datetime(
        df[date_col].astype(str) + " " + df[time_col].fillna("00:00").astype(str),
        format="%d-%m-%y %H:%M",
        errors="coerce"
    )

    if drop_original:
        df.drop(columns=[date_col, time_col], inplace=True)

    if verbose:
        print(
            f"[build_statement_datetime] created '{out_col}' "
            f"valid={df[out_col].notna().sum():,}/{len(df):,}"
        )

    return df

In [93]:
# Clean unamed column
df_3557 = df_3557.loc[:, ~df_3557.columns.str.contains("^Unnamed")]

In [94]:
df_3557 = build_statement_datetime(df_3557, drop_original=True, out_col="วันที่/เวลา")

[build_statement_datetime] created 'วันที่/เวลา' valid=138/138


In [96]:
col = "วันที่/เวลา"

df_3557 = df_3557[[col] + [c for c in df_3557.columns if c != col]]

In [97]:
source_configs = [
    {
        "df_source": df_pimas,
        "source_name": "purchases",
        "source_amount_col": "AFTERTAX",
        "statement_amount_col": "ถอนเงิน",
        "extra_source_cols": ["ACCTNO", "ACCTNAME"],
        "source_date_col": "BILLDATE",
        "max_matches": 3,                      # optional
        "out_col": "purchase_candidates",       # optional
    },
    {
        "df_source": df_pvmas,
        "source_name": "vouchers",
        "source_amount_col": "NETAMT",
        "source_billno_col": "VOUCNO",
        "statement_amount_col": "ถอนเงิน",
        "extra_source_cols": ["ACCTNO", "ACCTNAME"],
        "source_date_col": "VOUCDATE",
        "max_matches": 3,                      # optional
        "out_col": "voucher_candidates",       # optional
    },
]

df_out_3557 = match_multiple_sources_candidates_by_amount(
    df_statement=df_3557,
    source_configs=source_configs,
    tolerance=1.0,
)

[match_billno_candidates_by_amount] matched_rows=71/138
[match_billno_candidates_by_amount] matched_rows=85/138


In [100]:
df_out_3557.columns

Index(['วันที่/เวลา', 'รายการ', 'ถอนเงิน', 'ฝากเงิน', 'ยอดคงเหลือ', 'ช่องทาง',
       'รายละเอียด', 'purchase_candidates', 'voucher_candidates'],
      dtype='object')

In [116]:
def combine_match_candidates(
    df,
    purchase_col="purchases.MATCH_CANDIDATES",
    voucher_col="vouchers.MATCH_CANDIDATES",
    out_col="MATCH_SUMMARY",
    drop_inputs=True,
):
    def build_text(row):
        purchase = row.get(purchase_col)
        voucher = row.get(voucher_col)

        parts = []

        if pd.notna(purchase):
            parts.append(f"เลขที่บิล:\n{purchase}")

        if pd.notna(voucher):
            parts.append(f"เลขที่ใบสำคัญ:\n{voucher}")

        return "\n\n".join(parts) if parts else pd.NA

    df = df.copy()
    df[out_col] = df.apply(build_text, axis=1)

    if drop_inputs:
        df = df.drop(columns=[purchase_col, voucher_col], errors="ignore")

    return df

In [117]:
df_out_3557_combined = combine_match_candidates(df_out_3557,
                                          purchase_col="purchase_candidates",
                                          voucher_col="voucher_candidates",)

In [121]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter


def export_reconciliation_excel(
    df: pd.DataFrame,
    filename: str = "reconciliation.xlsx",
    *,
    match_col: str = "MATCH_SUMMARY",
    rename_match_col_to: str = "เลขที่บิล/เลขที่ใบสำคัญ",
    add_status_col: bool = True,
    status_col_name: str = "สถานะ",
):
    """
    Export reconciliation dataframe to nicely formatted Excel.

    Features:
    - renames MATCH_SUMMARY -> Thai header
    - fills NA with "-"
    - wraps match summary text
    - vertically top-aligns all cells
    - adds thin borders
    - colors matched rows in 2-tone green
    - colors unmatched rows in 2-tone red
    - optional status column for easier filter
    - freeze header + autofilter
    """

    df = df.copy()

    # rename match summary column
    if match_col in df.columns and rename_match_col_to != match_col:
        df = df.rename(columns={match_col: rename_match_col_to})

    final_match_col = rename_match_col_to if rename_match_col_to in df.columns else match_col

    # optional status column
    if add_status_col and final_match_col in df.columns:
        df[status_col_name] = df.apply(
            lambda r:
                "-" if r.get("ถอนเงิน") in ["-", None] else
                "✓ matched" if pd.notna(r.get(final_match_col)) and str(r.get(final_match_col)).strip() not in ["", "-", "<NA>"]
                else "✗ unmatched",
            axis=1
        )
        # place status right before match column
        cols = df.columns.tolist()
        cols.remove(status_col_name)
        match_idx = cols.index(final_match_col)
        cols = cols[:match_idx] + [status_col_name] + cols[match_idx:]
        df = df[cols]

    # fill NA for Excel
    df = df.fillna("-")

    wb = Workbook()
    ws = wb.active
    ws.title = "Bank Match"

    # write dataframe
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)

    # ---------- styles ----------
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(bold=True, color="FFFFFF")

    match_dark = PatternFill("solid", fgColor="C6E0B4")
    match_light = PatternFill("solid", fgColor="E2F0D9")

    unmatch_dark = PatternFill("solid", fgColor="F4CCCC")
    unmatch_light = PatternFill("solid", fgColor="FDE9E7")

    neutral_fill = PatternFill("solid", fgColor="F2F2F2")

    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"),
        right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"),
        bottom=Side(style="thin", color="D9D9D9"),
    )

    # header styling
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # freeze + filter
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    # locate special columns
    match_col_idx = list(df.columns).index(final_match_col) + 1 if final_match_col in df.columns else None
    status_col_idx = list(df.columns).index(status_col_name) + 1 if status_col_name in df.columns else None

    # body styling
    for row_idx in range(2, ws.max_row + 1):
        withdraw_value = ws.cell(row=row_idx, column=list(df.columns).index("ถอนเงิน")+1).value
        match_value = ws.cell(row=row_idx, column=match_col_idx).value

        is_withdraw = withdraw_value not in [None, "", "-", "<NA>"]
        is_matched = match_value not in [None, "", "-", "<NA>"]

        # alternating shade inside matched/unmatched groups
        alt = (row_idx % 2 == 0)
        if not is_withdraw:
            row_fill = neutral_fill
        elif is_matched:
            row_fill = match_dark if alt else match_light
        else:
            row_fill = unmatch_dark if alt else unmatch_light

        # row height based on wrapped lines in summary cell
        match_text = "" if match_value in [None, "-"] else str(match_value)
        line_count = max(1, match_text.count("\n") + 1)
        ws.row_dimensions[row_idx].height = max(22, line_count * 15)

        for col_idx in range(1, ws.max_column + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.fill = row_fill
            cell.border = thin_border

            if col_idx == match_col_idx:
                cell.alignment = Alignment(
                    horizontal="left",
                    vertical="top",
                    wrap_text=True
                )
            else:
                cell.alignment = Alignment(
                    horizontal="left",
                    vertical="top",
                    wrap_text=False
                )

    # number/date formatting + widths
    for col_idx, col_name in enumerate(df.columns, 1):
        col_letter = get_column_letter(col_idx)

        # date-like columns
        if col_name in ["วันที่/เวลา", "date", "DATE", "DATETIME"]:
            for cell in ws[col_letter][1:]:
                if isinstance(cell.value, pd.Timestamp):
                    cell.number_format = "yyyy-mm-dd hh:mm"
                elif cell.value not in ["-", None]:
                    # leave as-is if already string
                    pass
            ws.column_dimensions[col_letter].width = 20

        # numeric columns
        if col_name in ["ถอนเงิน", "ฝากเงิน", "ยอดคงเหลือ", "Amount"]:
            for cell in ws[col_letter][1:]:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = "#,##0.00"
            ws.column_dimensions[col_letter].width = 14

        # status column
        elif col_name == status_col_name:
            ws.column_dimensions[col_letter].width = 14

        # summary column
        elif col_name == final_match_col:
            ws.column_dimensions[col_letter].width = 45

        # long text details
        elif col_name in ["รายละเอียด", "Details", "DESCRIPTION"]:
            ws.column_dimensions[col_letter].width = 35

        # general columns
        elif col_name in ["รายการ", "ช่องทาง"]:
            ws.column_dimensions[col_letter].width = 18

        else:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in ws[col_letter])
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 25)

    wb.save(filename)

In [122]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "04_Statement_Report",
    f"statement_report_3557.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [123]:
export_reconciliation_excel(df_out_3557_combined, folder)